# Comparative SMOTE: Descriptive and Statistical Analysis (v2)

This notebook **does not retrain the models**. It loads exactly two source files: `comparative_smote_fold_results.csv` and `comparative_smote_resampling_counts.csv`, then performs:
1. An analysis of class counts before and after resampling.
2. Performance summaries by variant and model: mean, standard deviation (SD), and descriptive 95% t-based confidence intervals (CIs).
3. Friedman tests for each model and metric.
4. Paired Wilcoxon signed-rank tests for each model and metric, with Holm correction applied within each comparison family.
5. Displays tables in the notebook and saves results as CSV files with the `_v2.csv` suffix.

**Important:** Statistical tests require paired scores from the same experimental units. The code pairs observations using `Repeat` and `Outer_Fold`, checks for duplicate or missing pairs, and records the number of units used. Cross-validation fold scores may be dependent; interpret the CIs and statistical tests in light of the validation design. Review the notes in the statistical-analysis sections before making inferential claims.

In [ ]:
# 0. Imports and configuration
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, wilcoxon, t

try:
    from IPython.display import display
except ImportError:
    display = print

# Directory containing the notebook and source CSV files.
# If the CSV files are stored elsewhere, set DATA_DIR explicitly.
DATA_DIR = Path.cwd()
OUTPUT_DIR = DATA_DIR / "comparative_smote_analysis_outputs_v2"
PLOT_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Exact source filenames are defined below.
FOLD_STEM = "comparative_smote_fold_results"
COUNTS_STEM = "comparative_smote_resampling_counts"

METRICS = [
    # "accuracy", 
    # "precision", 
    # "recall", 
    # "specificity", 
    # "f1",
    # "roc_auc", 
    # "average_precision", 
    "pr_auc",
    "balanced_accuracy", 
    "mcc"
]

# Model and variant order are inferred from the data,
# unless manual lists are provided below.
MODEL_ORDER = None
VARIANT_ORDER = None

ALPHA = 0.05
N_BOOTSTRAP = 10000  # not used in the main analysis; retained for possible extensions
RANDOM_SEED = 42

print("DATA_DIR:", DATA_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## 1. Load the two source CSV files

This notebook reads only these exact filenames from `DATA_DIR`:
- `comparative_smote_fold_results.csv`
- `comparative_smote_resampling_counts.csv`

The files are not combined with numbered copies or other similarly named files. Duplicate rows are not silently removed from the source files.

In [ ]:
def load_source_csv(filename: str, required_columns=None):
    """Load one exact CSV filename without merging numbered copies."""
    file_path = DATA_DIR / filename
    if not file_path.is_file():
        raise FileNotFoundError(
            f"Required source file not found: {file_path}. "
            "Place the exact file in DATA_DIR or update DATA_DIR."
        )

    df = pd.read_csv(file_path)
    df.columns = [str(column).strip() for column in df.columns]
    df["_source_file"] = file_path.name
    print(f"Loaded {file_path.name}: {len(df):,} rows, {len(df.columns) - 1} data columns")

    if required_columns:
        missing = [column for column in required_columns if column not in df.columns]
        if missing:
            raise ValueError(f"Required columns missing from {filename}: {missing}")
    return df, file_path


fold_df, fold_file = load_source_csv(
    "comparative_smote_fold_results.csv",
    required_columns=["Variant", "Model", "Outer_Fold", "Repeat", "TN", "FP", "FN", "TP"]
)

counts_df, counts_file = load_source_csv(
    "comparative_smote_resampling_counts.csv",
    required_columns=[
        "Variant", "Model", "Outer_Fold", "Repeat",
        "Before_Class_0", "Before_Class_1",
        "After_Class_0", "After_Class_1"
    ]
)

# Normalize metric column names to lowercase, preserving other names.
fold_df.columns = [column.strip() for column in fold_df.columns]
for metric in METRICS:
    if metric not in fold_df.columns:
        # Case-insensitive fallback.
        candidates = [column for column in fold_df.columns if column.lower() == metric.lower()]
        if candidates:
            fold_df = fold_df.rename(columns={candidates[0]: metric})

available_metrics = [metric for metric in METRICS if metric in fold_df.columns]
if not available_metrics:
    raise ValueError(
        f"None of the configured metrics were found in comparative_smote_fold_results.csv. "
        f"Configured metrics: {METRICS}"
    )

print("\nModels:", sorted(fold_df["Model"].dropna().astype(str).unique()))
print("Variants:", sorted(fold_df["Variant"].dropna().astype(str).unique()))
print("Available metrics:", available_metrics)
display(fold_df.head())


## 2. Before-and-after resampling analysis

The before-and-after class-count analysis uses `comparative_smote_resampling_counts.csv` only. If this exact file is missing, the notebook raises an error rather than substituting count columns from the fold-results file. The code does not assume that every variant must produce balanced class counts.

In [ ]:
def ratio_majority_minority(majority, minority):
    majority = pd.to_numeric(majority, errors="coerce")
    minority = pd.to_numeric(minority, errors="coerce")
    return np.where(minority > 0, majority / minority, np.nan)


# Use the dedicated resampling-counts source file only.
resampling_source = counts_df.copy()
resampling_source["Majority_Before"] = resampling_source["Before_Class_0"]
resampling_source["Minority_Before"] = resampling_source["Before_Class_1"]
resampling_source["Majority_After"] = resampling_source["After_Class_0"]
resampling_source["Minority_After"] = resampling_source["After_Class_1"]

if "Before_Total" not in resampling_source.columns:
    resampling_source["Before_Total"] = (
        resampling_source["Before_Class_0"] + resampling_source["Before_Class_1"]
    )
if "After_Total" not in resampling_source.columns:
    resampling_source["After_Total"] = (
        resampling_source["After_Class_0"] + resampling_source["After_Class_1"]
    )

if resampling_source is not None:
    resampling_source["Ratio_Before_Majority_Minority"] = ratio_majority_minority(
        resampling_source["Majority_Before"], resampling_source["Minority_Before"]
    )
    resampling_source["Ratio_After_Majority_Minority"] = ratio_majority_minority(
        resampling_source["Majority_After"], resampling_source["Minority_After"]
    )

    resampling_summary = (
        resampling_source.groupby(["Variant", "Model"], dropna=False)
        .agg(
            N_Folds=("Outer_Fold", "count"),
            Majority_Before_Mean=("Majority_Before", "mean"),
            Minority_Before_Mean=("Minority_Before", "mean"),
            Total_Before_Mean=("Before_Total", "mean"),
            Majority_After_Mean=("Majority_After", "mean"),
            Minority_After_Mean=("Minority_After", "mean"),
            Total_After_Mean=("After_Total", "mean"),
            Ratio_Before_Mean=("Ratio_Before_Majority_Minority", "mean"),
            Ratio_After_Mean=("Ratio_After_Majority_Minority", "mean"),
        )
        .reset_index()
    )

    print("\n=== BEFORE–AFTER RESAMPLING SUMMARY ===")
    display(resampling_summary.round(4))
    resampling_summary.to_csv(OUTPUT_DIR / "resampling_counts_summary_v2.csv", index=False)

    # Plot mean class counts before and after, faceted by model and variant
    plot_counts = resampling_summary.melt(
        id_vars=["Variant", "Model"],
        value_vars=[
            "Majority_Before_Mean", "Minority_Before_Mean",
            "Majority_After_Mean", "Minority_After_Mean"
        ],
        var_name="Class_Status", value_name="Mean_Count"
    )
    plot_counts["Class_Status"] = plot_counts["Class_Status"].str.replace("_Mean", "", regex=False)
    display(plot_counts.head())
    plot_counts.to_csv(OUTPUT_DIR / "resampling_counts_plot_data_v2.csv", index=False)

    for model in sorted(resampling_summary["Model"].dropna().unique()):
        sub = resampling_summary[resampling_summary["Model"] == model]
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(max(10, len(sub) * 1.2), 6))
        x = np.arange(len(sub))
        width = 0.2
        series = [
            ("Majority_Before_Mean", "Majority before"),
            ("Minority_Before_Mean", "Minority before"),
            ("Majority_After_Mean", "Majority after"),
            ("Minority_After_Mean", "Minority after"),
        ]
        for i, (col, label) in enumerate(series):
            ax.bar(x + (i - 1.5) * width, sub[col], width, label=label)
        ax.set_xticks(x)
        ax.set_xticklabels(sub["Variant"].astype(str), rotation=35, ha="right")
        ax.set_title(f"Resampling Counts — {model}", fontsize=14)
        ax.set_ylabel("Mean number of training samples")
        ax.tick_params(axis="both", labelsize=11)
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)
        ax.legend()
        fig.tight_layout()
        out = PLOT_DIR / f"resampling_counts_{re.sub(r'[^A-Za-z0-9_-]+','_',str(model))}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print("Figure saved:", out)
else:
    resampling_summary = pd.DataFrame()


In [ ]:
# Overall before-and-after resampling summary,
# without separating by model.

resampling_summary_overall = (
    resampling_source.groupby("Variant", dropna=False)
    .agg(
        N_Folds=("Outer_Fold", "count"),

        Majority_Before_Mean=("Majority_Before", "mean"),
        Minority_Before_Mean=("Minority_Before", "mean"),
        Total_Before_Mean=("Before_Total", "mean"),

        Majority_After_Mean=("Majority_After", "mean"),
        Minority_After_Mean=("Minority_After", "mean"),
        Total_After_Mean=("After_Total", "mean"),

        Ratio_Before_Mean=("Ratio_Before_Majority_Minority", "mean"),
        Ratio_After_Mean=("Ratio_After_Majority_Minority", "mean"),
    )
    .reset_index()
)

print("\n=== BEFORE–AFTER RESAMPLING SUMMARY (ALL MODELS) ===")
display(resampling_summary_overall.round(4))

# Save results
resampling_summary_overall.to_csv(
    OUTPUT_DIR / "resampling_counts_summary_overall_v2.csv",
    index=False
)

print("File saved:", OUTPUT_DIR / "resampling_counts_summary_overall_v2.csv")

## 3. Performance summary: mean, SD, and 95% CI

The confidence interval is calculated as a t-based interval for the mean: `mean ± t_(0.975, n-1) × SD/sqrt(n)`. Because cross-validation scores may be dependent, the CI columns are labelled descriptive and should not automatically be interpreted as inferential CIs that account for fold dependence.

In [ ]:
def mean_sd_ci(values):
    x = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    n = len(x)
    if n == 0:
        return {"N_Scores": 0, "Mean": np.nan, "SD": np.nan,
                "CI95_Lower_Descriptive": np.nan, "CI95_Upper_Descriptive": np.nan}
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1)) if n > 1 else np.nan
    if n > 1:
        margin = t.ppf(0.975, df=n-1) * sd / np.sqrt(n)
        lower, upper = mean - margin, mean + margin
    else:
        lower, upper = np.nan, np.nan
    return {
        "N_Scores": n, "Mean": mean, "SD": sd,
        "CI95_Lower_Descriptive": lower, "CI95_Upper_Descriptive": upper
    }


summary_rows = []
for (variant, model), group in fold_df.groupby(["Variant", "Model"], dropna=False):
    for metric in available_metrics:
        stats = mean_sd_ci(group[metric])
        summary_rows.append({
            "Variant": variant, "Model": model, "Metric": metric, **stats,
            # "CI_Note": "Descriptive t-based CI; CV fold scores may be dependent"
        })

performance_summary = pd.DataFrame(summary_rows)

if VARIANT_ORDER is not None:
    performance_summary["Variant"] = pd.Categorical(
        performance_summary["Variant"], categories=VARIANT_ORDER, ordered=True
    )
if MODEL_ORDER is not None:
    performance_summary["Model"] = pd.Categorical(
        performance_summary["Model"], categories=MODEL_ORDER, ordered=True
    )

print("\n=== PERFORMANCE SUMMARY: MEAN, SD, 95% CI ===")
display(performance_summary.round(5))
performance_summary.to_csv(OUTPUT_DIR / "performance_summary_v2.csv", index=False)

# Also print a compact Mean ± SD [95% CI] table for quick reading
compact = performance_summary.copy()
compact["Mean ± SD [95% CI]"] = compact.apply(
    lambda r: (
        f'{r["Mean"]:.4f} ± {r["SD"]:.4f} '
        f'[{r["CI95_Lower_Descriptive"]:.4f}, {r["CI95_Upper_Descriptive"]:.4f}]'
        if pd.notna(r["SD"]) and pd.notna(r["CI95_Lower_Descriptive"])
        else f'{r["Mean"]:.4f} (CI unavailable)'
        if pd.notna(r["Mean"]) else "NA"
    ),
    axis=1
)
print("\n=== COMPACT FORMAT FOR THE MANUSCRIPT ===")
display(compact[["Variant", "Model", "Metric", "N_Scores", "Mean ± SD [95% CI]"]])
compact[["Variant", "Model", "Metric", "N_Scores", "Mean ± SD [95% CI]"]].to_csv(
    OUTPUT_DIR / "performance_summary_compact_v2.csv", index=False
)


## 4. Bar charts by model and metric

One figure is generated for each metric, with separate panels for each model. Bars show the mean, and error bars show the bounds of the descriptive 95% t-based CI. The y-axis is adjusted for each metric so the displayed range is appropriate to that metric, without automatically truncating the scale to exaggerate differences.

In [ ]:
for metric in available_metrics:
    summary_metric = performance_summary[performance_summary["Metric"] == metric].copy()
    if summary_metric.empty:
        continue

    models = list(MODEL_ORDER) if MODEL_ORDER is not None else sorted(summary_metric["Model"].dropna().astype(str).unique())
    variants = list(VARIANT_ORDER) if VARIANT_ORDER is not None else sorted(summary_metric["Variant"].dropna().astype(str).unique())

    ncols = 2
    nrows = int(np.ceil(len(models) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 5.5 * nrows), squeeze=False)
    axes = axes.flatten()

    for ax, model in zip(axes, models):
        sub = summary_metric[summary_metric["Model"].astype(str) == str(model)].set_index("Variant")
        sub = sub.reindex([v for v in variants if v in sub.index])
        if sub.empty:
            ax.axis("off")
            continue

        means = sub["Mean"].to_numpy(dtype=float)
        lower = sub["CI95_Lower_Descriptive"].to_numpy(dtype=float)
        upper = sub["CI95_Upper_Descriptive"].to_numpy(dtype=float)
        yerr_low = np.where(np.isfinite(lower), np.maximum(0, means - lower), 0)
        yerr_up = np.where(np.isfinite(upper), np.maximum(0, upper - means), 0)
        yerr = np.vstack([yerr_low, yerr_up])
        x = np.arange(len(sub))

        ax.bar(
            x, means, yerr=yerr, capsize=4,
            edgecolor="black", linewidth=0.7, alpha=0.85
        )
        ax.set_xticks(x)
        ax.set_xticklabels([str(v) for v in sub.index], rotation=30, ha="right", fontsize=10)
        ax.tick_params(axis="y", labelsize=10)
        ax.set_title(str(model), fontsize=13)
        ax.set_ylabel(metric.replace("_", " ").title(), fontsize=11)
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)

        if metric.lower() != "mcc" and means.size and np.nanmin(means) >= 0 and np.nanmax(means) <= 1:
            ax.set_ylim(0, min(1.15, max(0.1, np.nanmax(upper[np.isfinite(upper)]) * 1.12 if np.isfinite(upper).any() else np.nanmax(means) * 1.12)))
        elif metric.lower() == "mcc":
            ax.axhline(0, linewidth=0.8)

    for ax in axes[len(models):]:
        ax.axis("off")

    fig.suptitle(f"Comparative SMOTE — {metric.replace('_', ' ').title()}", fontsize=16, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    out = PLOT_DIR / f"performance_{metric}_v2.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Figure saved:", out)


## 5. Preparing paired observations for Friedman and Wilcoxon tests

Tests are conducted separately for each model and metric. Each variant must have scores for the same experimental units. The code uses `Repeat` + `Outer_Fold` as pairing keys and reports duplicate combinations or incomplete pairs. If multiple rows share the same pairing unit, the code stops the affected comparison rather than silently averaging them.

**Design warning:** Cross-validation folds are not independent observations. Friedman/Wilcoxon tests on paired outer-fold results may be useful for the requested analysis, but conventional p-values do not automatically correct for dependence caused by overlapping training sets. Interpret results with this limitation in mind and follow the statistical design specified in the research methodology.

In [ ]:
PAIR_KEYS = ["Repeat", "Outer_Fold"]
for col in PAIR_KEYS:
    if col not in fold_df.columns:
        raise ValueError(f"Pairing column {col} is missing. Check the fold-results structure.")

# Convert keys and metrics to numeric where appropriate.
paired_df = fold_df.copy()
for col in PAIR_KEYS:
    paired_df[col] = pd.to_numeric(paired_df[col], errors="coerce")
for metric in available_metrics:
    paired_df[metric] = pd.to_numeric(paired_df[metric], errors="coerce")

duplicate_key_rows = paired_df.duplicated(
    subset=["Model", "Variant"] + PAIR_KEYS, keep=False
)
if duplicate_key_rows.any():
    print("WARNING: duplicate rows found for Model × Variant × Repeat × Outer_Fold.")
    display(paired_df.loc[duplicate_key_rows, ["Model", "Variant"] + PAIR_KEYS + available_metrics].head(20))
    print("To avoid ambiguous pairing, duplicate rows are not averaged automatically. Clean the data or define the analysis unit before interpreting statistical tests.")

def make_paired_wide(data, model, metric):
    sub = data.loc[data["Model"].astype(str) == str(model), ["Variant"] + PAIR_KEYS + [metric]].copy()
    sub = sub.dropna(subset=["Variant"] + PAIR_KEYS + [metric])
    if sub.duplicated(subset=["Variant"] + PAIR_KEYS).any():
        return None, "Duplicate scores for a Variant × Repeat × Outer_Fold key; pairing is ambiguous."
    wide = sub.pivot(index=PAIR_KEYS, columns="Variant", values=metric)
    return wide, None

models_for_tests = list(MODEL_ORDER) if MODEL_ORDER is not None else sorted(paired_df["Model"].dropna().astype(str).unique())
variants_for_tests = list(VARIANT_ORDER) if VARIANT_ORDER is not None else sorted(paired_df["Variant"].dropna().astype(str).unique())
print("Models to be tested:", models_for_tests)
print("Variants to be tested:", variants_for_tests)
print("Pairing keys:", PAIR_KEYS)


## 6. Friedman test by model and metric

The Friedman test is performed only when at least three variants have paired scores on the same units. Units with incomplete data are removed listwise for the relevant model and metric. The number of complete units is recorded.

In [ ]:
friedman_rows = []

for model in models_for_tests:
    for metric in available_metrics:
        wide, issue = make_paired_wide(paired_df, model, metric)
        if issue:
            friedman_rows.append({
                "Model": model, "Metric": metric, "Test": "Friedman",
                "Statistic": np.nan, "P_Value": np.nan, "N_Complete_Paired_Units": 0,
                "N_Variants": 0, "Variants": "", "Status": issue
            })
            continue

        # Keep only variants present in this model/metric, then complete paired rows.
        variant_cols = [v for v in variants_for_tests if v in wide.columns]
        complete = wide[variant_cols].dropna() if variant_cols else pd.DataFrame()
        if len(variant_cols) < 3:
            status = "Skipped: fewer than 3 variants available."
            stat, pvalue = np.nan, np.nan
        elif len(complete) < 2:
            status = "Skipped: insufficient complete paired units."
            stat, pvalue = np.nan, np.nan
        else:
            arrays = [complete[v].to_numpy(dtype=float) for v in variant_cols]
            try:
                stat, pvalue = friedmanchisquare(*arrays)
                status = "OK"
            except Exception as exc:
                stat, pvalue = np.nan, np.nan
                status = f"Error: {exc}"

        friedman_rows.append({
            "Model": model, "Metric": metric, "Test": "Friedman",
            "Statistic": stat, "P_Value": pvalue,
            "N_Complete_Paired_Units": len(complete),
            "N_Variants": len(variant_cols),
            "Variants": "; ".join(map(str, variant_cols)),
            "Status": status
        })

friedman_results = pd.DataFrame(friedman_rows)
print("\n=== FRIEDMAN TEST RESULTS BY MODEL AND METRIC ===")
display(friedman_results.round(6))
friedman_results.to_csv(OUTPUT_DIR / "friedman_results_v2.csv", index=False)


## 7. Pairwise Wilcoxon signed-rank tests by variant, model, and metric

All available variant pairs are tested using paired observations from the same experimental units. Holm correction is applied to all pairs within each `Model × Metric` comparison family. The results include the number of pairs actually used.

In [ ]:
from itertools import combinations

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(p.shape, np.nan, dtype=float)
    valid_idx = np.where(np.isfinite(p))[0]
    if len(valid_idx) == 0:
        return adjusted
    order = valid_idx[np.argsort(p[valid_idx])]
    m = len(order)
    running_max = 0.0
    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * p[idx])
        running_max = max(running_max, value)
        adjusted[idx] = running_max
    return adjusted


wilcoxon_rows = []

for model in models_for_tests:
    for metric in available_metrics:
        wide, issue = make_paired_wide(paired_df, model, metric)
        if issue:
            wilcoxon_rows.append({
                "Model": model, "Metric": metric, "Variant_A": None, "Variant_B": None,
                "Statistic": np.nan, "P_Value": np.nan, "P_Value_Holm": np.nan,
                "N_Paired_Units": 0, "Status": issue
            })
            continue

        variant_cols = [v for v in variants_for_tests if v in wide.columns]
        raw_results = []

        for a, b in combinations(variant_cols, 2):
            pair = wide[[a, b]].dropna()
            x = pair[a].to_numpy(dtype=float)
            y = pair[b].to_numpy(dtype=float)

            if len(pair) < 2:
                stat, pvalue = np.nan, np.nan
                status = "Skipped: fewer than 2 paired units."
            elif np.allclose(x - y, 0):
                stat, pvalue = 0.0, 1.0
                status = "All paired differences are zero."
            else:
                try:
                    stat, pvalue = wilcoxon(x, y, alternative="two-sided", zero_method="wilcox", method="auto")
                    status = "OK"
                except Exception as exc:
                    stat, pvalue = np.nan, np.nan
                    status = f"Error: {exc}"

            raw_results.append({
                "Model": model, "Metric": metric, "Variant_A": a, "Variant_B": b,
                "Statistic": stat, "P_Value": pvalue, "N_Paired_Units": len(pair),
                "Status": status
            })

        if raw_results:
            adjusted = holm_adjust([r["P_Value"] for r in raw_results])
            for row, p_adj in zip(raw_results, adjusted):
                row["P_Value_Holm"] = p_adj
                row["Significant_Holm_0.05"] = bool(p_adj < ALPHA) if np.isfinite(p_adj) else False
                wilcoxon_rows.append(row)

wilcoxon_results = pd.DataFrame(wilcoxon_rows)
print("\n=== WILCOXON TEST RESULTS BY MODEL, METRIC, AND VARIANT PAIR ===")

# Display all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

display(wilcoxon_results.round(6))
wilcoxon_results.to_csv(OUTPUT_DIR / "wilcoxon_results_v2.csv", index=False)


## 8. Audit summary and output files

This section reports the input status, data size after loading, available metrics, and output files. Check that the results are consistent with the expected number of experiments before using any values in the manuscript.

In [ ]:
print("=== DATA AUDIT ===")
print(f"Fold-results source file: {fold_file.name}")
print(f"Resampling-counts source file: {counts_file.name}")
print(f"Rows in fold results: {len(fold_df):,}")
print(f"Rows in resampling counts: {len(counts_df):,}")
print(f"Models: {fold_df['Model'].nunique()}")
print(f"Variants: {fold_df['Variant'].nunique()}")
print(f"Available metrics: {available_metrics}")
print(f"Rows in performance summary: {len(performance_summary):,}")
print(f"Rows in Friedman results: {len(friedman_results):,}")
print(f"Rows in Wilcoxon results: {len(wilcoxon_results):,}")

print("\\n=== OUTPUT FILES ===")
for output_path in sorted(OUTPUT_DIR.glob("*")):
    print(output_path.name)
print("\\nFigure files:")
for figure_path in sorted(PLOT_DIR.glob("*")):
    print(figure_path.name)

print("\\nCompleted. Review the pairing and CI notes before interpreting p-values.")
